In [31]:
import sqlite3
import pandas as pd

# Connect to database created in previous notebook
conn = sqlite3.connect('fred.db')

## Query 1 -- Highest Average Indicator During COVID

In [32]:
query = """
SELECT
    series_metadata.series_name,
    AVG(fred_monthly_long.value) AS avg_value
FROM fred_monthly_long
JOIN period_lookup
    ON fred_monthly_long.year = period_lookup.year
JOIN series_metadata
    ON fred_monthly_long.series_id = series_metadata.series_id
WHERE period_lookup.period = 'COVID'
GROUP BY series_metadata.series_name
ORDER BY avg_value DESC;
"""

output = pd.read_sql(query, conn)
output

# The first JOIN connects fred_monthly_long with the period_lookup table using the year column
# The second JOIN connects fred_monthly_long with the series_metadata table using the series_id column
# WHERE filters the dataset so only rows from the COVID period are included
# AVG() calculates the average value for each indicator
# GROUP BY groups all observations in the same indicator together so averages can be calculated separately for each category
# ORDER BY sorts the results from highest average value to lowest average value


,series_name,avg_value
0,Revolving Credit,990231.317500
1,Retail Sales,555022.208333
2,Real Disposable Income,16940.445833
3,Housing Starts,1498.833333
4,CPI,264.914583
5,Home Price Index,241.025375
6,PCE Price Index,106.819500
7,Consumer Sentiment,79.579167
8,Hourly Earnings,25.297083
9,Personal Savings Rate,13.208333


This query analyzes how different economic indicators behaved during the COVID-19 period. The goal is to identify which indicators maintained the highest average levels throughout the pandemic era. This can help reveal which parts of the economy remained large, stable, or heavily impacted during a major economic disruption. By limiting the analysis to the COVID period only, the query isolates economic conditions specifically associated with the pandemic rather than long-term historical trends.

The results show that Revolving Credit and Retail Sales had the highest average values during the COVID period. This suggests that consumer borrowing and spending remained major components of economic activity throughout the pandemic. Real Disposable Income also remained relatively high during COVID. This may have been influenced by government stimulus programs and relief payments that temporarily increased household income during the pandemic. In contrast, the Fed Funds Rate had the lowest average value during the COVID period. This reflects the Federal Reserve’s decision to lower interest rates to near zero in order to support borrowing and economic recovery during the pandemic.



## Query 2 -- Monthly Average Across All Indicators

In [33]:
query = """
SELECT
    month,
    AVG(value) AS avg_monthly_value
FROM fred_monthly_long
GROUP BY month
ORDER BY avg_monthly_value DESC;
"""

output = pd.read_sql(query, conn)
output

# AVG() computes the overall average economic value for each month
# GROUP BY month organizes observations into monthly groups
# ORDER BY sorts the months from highest average value to lowest average value

,month,avg_monthly_value
0,11,138361.881775
1,12,138181.985767
2,10,138061.366067
3,9,137486.195017
4,8,136799.444533
5,7,136230.190092
6,6,135609.260508
7,5,134608.585475
8,3,134089.961900
9,4,133647.829808


This query examines the average economic indicator value for each month across the entire dataset. The results reveal a very clear upward trend as the year progresses, with average values steadily increasing from January through November before slightly declining in December. November has the highest average monthly value at approximately 138,362, while January has the lowest at approximately 132,990.Economic activity appears to strengthen gradually during the middle and later portions of the year. Consumer spending generally rises during the second half of the year, especially approaching the holiday season.

This query demonstrates that seasonality plays an important role in economic behavior within the dataset. The findings suggest that economic indicators generally strengthen throughout the year, peak during the fall and early winter months, and begin at lower levels early in the calendar year. These seasonal trends are valuable because they help explain recurring patterns in economic performance and can assist in forecasting future economic conditions.

## Query 3 -- Rolling Average of Economic Indicators

In [34]:
query = """
SELECT
    year,
    month,
    series_id,
    value,
    AVG(value) OVER (
        PARTITION BY series_id
        ORDER BY year, month
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ) AS rolling_avg
FROM fred_monthly_long
ORDER BY series_id, year, month;
"""

output = pd.read_sql(query, conn)
output

# AVG() OVER() is a window function that performs calculations across related rows
# PARTITION BY series_id keeps each indicator separate during calculations
# ORDER BY year and month ensures the rolling average follows chronological order
# ROWS BETWEEN 2 PRECEDING AND CURRENT ROW creates a rolling window of three observations

,year,month,series_id,value,rolling_avg
0,2015,1,AHETPI,20.80,20.800000
1,2015,2,AHETPI,20.84,20.820000
2,2015,3,AHETPI,20.91,20.850000
3,2015,4,AHETPI,20.92,20.890000
4,2015,5,AHETPI,20.98,20.936667
...,...,...,...,...,...
1435,2024,8,UNRATE,4.20,4.166667
1436,2024,9,UNRATE,4.10,4.166667
1437,2024,10,UNRATE,4.10,4.133333
1438,2024,11,UNRATE,4.20,4.133333


This query calculates a rolling three-period average for each economic indicator in the dataset. A rolling average smooths short-term fluctuations by averaging the current observation with the two previous observations. This is useful to get around potential temporary volatility that can obscure larger trends.

For example, in late 2024, the unemployment rate fluctuates between 4.1% and 4.2%, but the rolling average remains relatively stable around 4.13% to 4.17%. This indicates that although small monthly changes occur, the overall labor market conditions remain fairly consistent during this period.

The query also shows the usefulness of window functions in economic analysis since they allow calculations to occur across related rows while still preserving individual observations.

## Query 4 -- Count of Observations by Economic Period

In [35]:
query = """
SELECT
    period_lookup.period,
    COUNT(*) AS observation_count
FROM fred_monthly_long
JOIN period_lookup
    ON fred_monthly_long.year = period_lookup.year
GROUP BY period_lookup.period
ORDER BY observation_count DESC;
"""

output = pd.read_sql(query, conn)
output

# JOIN connects the observations table with the period classification table
# COUNT(*) calculates the total number of rows in each economic period
# GROUP BY separates the counts by period
# ORDER BY sorts the periods from largest count to smallest count

,period,observation_count
0,Pre-COVID,720
1,Post-COVID,432
2,COVID,288


This query measures the number of observations contained within each major economic period: Pre-COVID, COVID, and Post-COVID. The results show that the dataset is not evenly distributed across periods, with the Pre-COVID era containing the largest number of observations by a substantial margin. 

An important implication of these results is that comparisons between periods should account for the unequal sample sizes. Since the Pre-COVID period contains substantially more observations, averages and trends from that era may appear more stable simply because they are based on more data points. In contrast, the COVID period may show greater volatility because fewer observations are concentrated during an unusually unstable economic environment.

## Query 5 -- Indicators Above Overall Average (Using a Subquery)

In [36]:
query = """
SELECT
    series_metadata.series_name,
    fred_monthly_long.value
FROM fred_monthly_long
JOIN series_metadata
    ON fred_monthly_long.series_id = series_metadata.series_id
WHERE fred_monthly_long.value > (
    SELECT AVG(value)
    FROM fred_monthly_long
)
ORDER BY fred_monthly_long.value DESC;
"""

output = pd.read_sql(query, conn)
output

# JOIN adds readable names to the output
# The subquery calculates the average value across the entire dataset
# WHERE filters rows so only values above the dataset average remain
# ORDER BY sorts the highest observations first

,series_name,value
0,Revolving Credit,1352433.61
1,Revolving Credit,1342480.89
2,Revolving Credit,1342090.49
3,Revolving Credit,1340184.49
4,Revolving Credit,1337484.66
...,...,...
235,Retail Sales,434470.00
236,Retail Sales,433647.00
237,Retail Sales,428208.00
238,Retail Sales,427119.00


This query identifies observations with values above the overall dataset average. It highlights which economic indicators consistently operate above average levels and helps identify the strongest contributors to overall economic performance. The results show that indicators such as Revolving Credit and Retail Sales appear frequently among the highest values in the dataset.

The dominance of Revolving Credit suggests that consumer borrowing plays a major role in overall economic activity. High revolving credit levels often indicate increased consumer spending and reliance on credit cards or short-term borrowing. Similarly, strong Retail Sales values reflect high levels of consumer demand and purchasing activity. These findings suggest that consumer behavior is a major driver of the economy within this dataset. 

## Query 6 -- Highest Savings Month Each Year (Using a Window Function + Subquery)

In [37]:
query = """
SELECT
    year,
    month,
    value AS savings_rate
FROM (
    SELECT
        year,
        month,
        value,
        DENSE_RANK() OVER (
            PARTITION BY year
            ORDER BY value DESC
        ) AS rank_num
    FROM fred_monthly_long
    WHERE series_id = 'PSAVERT'
)
WHERE rank_num = 1
ORDER BY year;
"""

output = pd.read_sql(query, conn)
output

# DENSE_RANK() is a window function that ranks rows within each year
# PARTITION BY year separates the ranking process by calendar year
# ORDER BY value DESC ranks the highest savings rates first
# The subquery creates ranked rows before the outer query filters the top-ranked month
# WHERE rank_num = 1 keeps only the highest savings month for each year

,year,month,savings_rate
0,2015,2,6.4
1,2016,1,6.1
2,2017,5,6.3
3,2018,12,8.4
4,2019,2,8.5
5,2020,4,31.8
6,2021,3,26.2
7,2022,1,4.2
8,2022,12,4.2
9,2023,5,6.1


This query identifies the highest personal savings rate month in every calendar year. It helps reveal how savings behavior changed over time and highlights periods where Americans saved at unusually high levels. The query is particularly useful for identifying the impact of economic shocks such as COVID-19.

The results show that 2020 and 2021 contained dramatically higher peak savings months than any pre-pandemic year. These spikes reflect stimulus payments, reduced consumer spending opportunities, and heightened economic uncertainty during the pandemic period. The query demonstrates how window functions can efficiently rank observations within grouped time periods.

## Query 7 -- Average Economic Indicators by Era (Using Multiple JOINs + GROUP BY)

In [38]:
query = """
SELECT
    period_lookup.period,
    series_metadata.series_name,
    ROUND(AVG(fred_monthly_long.value), 2) AS avg_value
FROM fred_monthly_long
JOIN period_lookup
    ON fred_monthly_long.year = period_lookup.year
JOIN series_metadata
    ON fred_monthly_long.series_id = series_metadata.series_id
GROUP BY
    period_lookup.period,
    series_metadata.series_name
ORDER BY
    period_lookup.period,
    avg_value DESC;
"""

output = pd.read_sql(query, conn)
output

# The first JOIN connects the observations table with the economic era lookup table
# The second JOIN connects observations with readable metadata labels
# AVG() calculates the average value for each indicator within each economic era
# GROUP BY separates the calculations by both era and indicator name
# ORDER BY organizes the output by era and descending average values

,period,series_name,avg_value
0,COVID,Revolving Credit,990231.32
1,COVID,Retail Sales,555022.21
2,COVID,Real Disposable Income,16940.45
3,COVID,Housing Starts,1498.83
4,COVID,CPI,264.91
5,COVID,Home Price Index,241.03
6,COVID,PCE Price Index,106.82
7,COVID,Consumer Sentiment,79.58
8,COVID,Hourly Earnings,25.30
9,COVID,Personal Savings Rate,13.21


This query compares economic indicators across the Pre-COVID, COVID, and Post-COVID periods. It provides a broad view of how macroeconomic conditions changed across different economic environments and allows direct comparison of average indicator levels across eras.

The results show clear structural differences between the three periods. Savings rates and unemployment rose sharply during COVID, while consumer activity indicators such as retail sales and revolving credit became more dominant in the Post-COVID era. The query highlights how economic conditions shifted throughout the decade and demonstrates the usefulness of joins for combining contextual metadata with raw observations.

## Query 8 -- Month-to-Month Savings Rate Change (LAG Window Function with 6-Month Filtered Output)

In [43]:
query = """
SELECT *
FROM (
    SELECT
        fred_data.year,
        fred_data.month,
        fred_data.value AS savings_rate,
        LAG(fred_data.value, 1) OVER (
            ORDER BY fred_data.year, fred_data.month
        ) AS previous_month,
        fred_data.value - LAG(fred_data.value, 1) OVER (
            ORDER BY fred_data.year, fred_data.month
        ) AS monthly_change
    FROM fred_monthly_long AS fred_data
    JOIN period_lookup AS period_table
        ON fred_data.year = period_table.year
    WHERE fred_data.series_id = 'PSAVERT'
) sub
WHERE (year * 12 + month) % 6 = 0
ORDER BY year, month;
"""

output = pd.read_sql(query, conn)
output

output = pd.read_sql(query, conn)
output

# LAG() is a window function that retrieves the previous month’s savings rate in chronological order
# ORDER BY year and month ensures observations are processed in time sequence
# The subtraction calculates the month-to-month change in savings rates
# JOIN with period_lookup is included to maintain consistency with other period-based analyses, even though no filter is applied
# The outer query reduces output by selecting every 6th month to make results more readable while preserving full window-function computation
# ORDER BY organizes the final output chronologically

,year,month,savings_rate,previous_month,monthly_change
0,2015,6,5.7,5.8,-0.1
1,2015,12,5.8,5.6,0.2
2,2016,6,4.9,5.3,-0.4
3,2016,12,5.0,5.4,-0.4
4,2017,6,6.0,6.3,-0.3
5,2017,12,5.0,5.6,-0.6
6,2018,6,6.3,6.0,0.3
7,2018,12,8.4,6.5,1.9
8,2019,6,7.1,7.3,-0.2
9,2019,12,6.2,6.9,-0.7


This query measures how personal savings rates evolve over time using a 6-month sampled view of month-to-month changes. Instead of displaying every observation, the output highlights broader structural movements while preserving the underlying LAG-based change calculation.

The results show a relatively stable savings environment prior to 2020, with savings rates generally fluctuating between roughly 5% and 7% and only small month-to-month changes. This indicates consistent household saving behavior during normal economic conditions.

A clear structural break occurs in 2020, where savings rates rise dramatically to above 18%, reflecting the economic disruption caused by the COVID-19 pandemic, reduced consumer spending, and stimulus effects. Even in the reduced 6-month sampling, this spike remains highly visible and represents the dominant feature of the dataset.

Following this shock, savings rates gradually normalize back toward pre-pandemic levels, stabilizing in the 5%–7% range by 2022–2024. Overall, the results suggest a temporary but extreme behavioral shift during COVID, followed by a reversion toward long-run historical averages.

## Query 9 -- Top 5 Highest Savings Months with Matching Unemployment Rate (Using a Correlated Subquery)

In [40]:
query = """
SELECT
    savings_data.year,
    savings_data.month,
    savings_data.value AS savings_rate,
    (
        SELECT unemployment_data.value
        FROM fred_monthly_long AS unemployment_data
        WHERE unemployment_data.series_id = 'UNRATE'
          AND unemployment_data.year = savings_data.year
          AND unemployment_data.month = savings_data.month
    ) AS unemployment_rate
FROM fred_monthly_long AS savings_data
WHERE savings_data.series_id = 'PSAVERT'
ORDER BY savings_data.value DESC
LIMIT 5;
"""

output = pd.read_sql(query, conn)
output

# savings_data represents the personal savings rate series (PSAVERT)
# unemployment_data represents the unemployment rate series (UNRATE)
# The correlated subquery matches unemployment to the same year and month as savings
# ORDER BY ranks months by highest savings rate
# LIMIT 5 restricts output to the top 5 savings months only
# This allows comparison of unemployment conditions during extreme savings periods

,year,month,savings_rate,unemployment_rate
0,2020,4,31.8,14.8
1,2021,3,26.2,6.1
2,2020,5,22.6,13.2
3,2021,1,19.5,6.4
4,2020,6,18.3,11.0


This query identifies the five highest personal savings rate months and matches each observation with the corresponding unemployment rate in the same time period. The correlated subquery retrieves unemployment values by aligning both series on year and month, ensuring a direct temporal comparison between savings behavior and labor market conditions.

The results highlight how unemployment behaves during extreme savings periods, such as during economic shocks or recessions. This structure demonstrates how correlated subqueries can be used not only for filtering within a series but also for joining related economic indicators without using a traditional JOIN operation.

## Query 10 -- Savings Rate and Unemployment Rate Comparison (Using a Self-JOIN)

In [41]:
query = """
SELECT
    savings.year,
    AVG(savings.value) AS average_savings_rate,
    AVG(unemployment.value) AS average_unemployment_rate
FROM fred_monthly_long AS savings
JOIN fred_monthly_long AS unemployment
    ON savings.year = unemployment.year
WHERE savings.series_id = 'PSAVERT'
    AND unemployment.series_id = 'UNRATE'
GROUP BY savings.year
ORDER BY savings.year;
"""

output = pd.read_sql(query, conn)
output

# The query uses a self-JOIN because both indicators exist in the same table
# The JOIN matches rows using the same year and month
# WHERE filters one side of the join to savings rates and the other to unemployment rates
# The query combines two economic indicators into one row for direct comparison
# ORDER BY organizes the observations chronologically

,year,average_savings_rate,average_unemployment_rate
0,2015,5.850000,5.275000
1,2016,5.358333,4.875000
2,2017,5.758333,4.358333
3,2018,6.433333,3.891667
4,2019,7.308333,3.675000
5,2020,15.091667,8.100000
6,2021,11.325000,5.350000
7,2022,3.350000,3.650000
8,2023,5.591667,3.625000
9,2024,5.433333,4.025000


This query compares personal savings rates and unemployment rates side-by-side across time. It allows direct observation of how labor market conditions relate to household saving behavior and helps identify periods where the relationship between the two variables changed significantly.

The results show that savings and unemployment both surged during the COVID period, especially in 2020. This pattern suggests that economic uncertainty and labor market disruptions contributed to unusually high household savings. In the Post-COVID period, unemployment normalized while savings rates declined substantially, indicating that households returned to more typical spending behavior even as labor markets improved.